# Rapport - Module 3 : agrégation et détection de force brute

Auteur : Youness - *premier jet*

Le module `log_engine/aggregation.py` consomme la sortie du module 2 (`transform`) **en un seul passage**, par blocs de taille bornée, sans jamais construire de liste complète des logs.

In [1]:
import os
import sys
from pathlib import Path

# Le notebook peut être lancé depuis notebooks/ : on se place à la racine du dépôt.
root = Path.cwd()
if not (root / "log_engine").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

In [2]:
from pathlib import Path
from log_engine.ingestion import parse_logs, read_multiple_logs
from log_engine.transform import transform
from log_engine.aggregation import analyze, format_report, compare_with_blacklist

paths = sorted(Path("data").glob("*.log"))
entries = transform(parse_logs(read_multiple_logs(paths)))  # itérateur paresseux
report = analyze(entries)
print(format_report(report))

Entrées analysées : 15 (analyse complète)
Par canal  : {'auth': 2, 'event': 2, 'auth_failure': 5, 'error': 3, 'raw': 3}
Par user   : {'alice': 2, 'bob': 1, 'root': 5, 'carol': 1, 'dave': 2, 'admin': 2, 'guest': 1, 'erin': 1}
Par niveau : {'INFO': 6, 'WARNING': 4, 'ERROR': 4, 'CRITICAL': 1}
IP en erreur       : ['10.0.0.5', '172.16.0.3', '203.0.113.7']
IP actives         : ['10.0.0.5', '172.16.0.3', '192.168.1.10', '192.168.1.11', '192.168.1.12', '192.168.1.13', '198.51.100.4', '203.0.113.7']
IP actives et blacklistées : ['198.51.100.4', '203.0.113.7']
IP actives non blacklistées : ['10.0.0.5', '172.16.0.3', '192.168.1.10', '192.168.1.11', '192.168.1.12', '192.168.1.13']
IP blacklistées inactives   : ['45.33.32.156']
Différence symétrique       : ['10.0.0.5', '172.16.0.3', '192.168.1.10', '192.168.1.11', '192.168.1.12', '192.168.1.13', '45.33.32.156']
ALERTE force brute : 10.0.0.5 - 5 échecs consécutifs (08:01:10 -> 08:01:18), users=['root']


## 1. Ensembles et unicité (`set`, `frozenset`)

- `error_ips` : `set` des IP ayant produit un niveau `ERROR` ou `CRITICAL`, converti en `frozenset` dans le rapport.
- `DEFAULT_BLACKLIST` est un `frozenset` : immuable, donc non modifiable par erreur.
- Les quatre opérateurs `|`, `&`, `-`, `^` sont appliqués entre IP actives et blacklist dans `compare_with_blacklist`.

In [3]:
cmp = compare_with_blacklist({"10.0.0.5", "203.0.113.7"}, frozenset({"203.0.113.7", "45.33.32.156"}))
for name, value in cmp._asdict().items():
    print(f"{name:22} {sorted(value)}")

union                  ['10.0.0.5', '203.0.113.7', '45.33.32.156']
intersection           ['203.0.113.7']
only_active            ['10.0.0.5']
only_blacklist         ['45.33.32.156']
symmetric_difference   ['10.0.0.5', '45.33.32.156']


## 2. Accumulation dynamique (`defaultdict`)

`defaultdict(int)` compte par canal, utilisateur et niveau sans jamais lever de `KeyError` : la première occurrence d'une clé démarre à 0. Le rapport expose ensuite de simples `dict`, pour qu'une lecture d'une clé absente n'en crée pas une.

In [4]:
print(report.by_channel)
print(report.by_user)
print(report.by_level)

{'auth': 2, 'event': 2, 'auth_failure': 5, 'error': 3, 'raw': 3}
{'alice': 2, 'bob': 1, 'root': 5, 'carol': 1, 'dave': 2, 'admin': 2, 'guest': 1, 'erin': 1}
{'INFO': 6, 'WARNING': 4, 'ERROR': 4, 'CRITICAL': 1}


## 3. Détection de force brute (`while`, `for`, `break`, `continue`, `else`)

Règle : **5 échecs de connexion consécutifs depuis une même IP**.

- Le `while True` boucle sur des blocs de `chunk_size` entrées (`islice`), et s'arrête quand le flux est épuisé (`break`).
- Le `for` parcourt un bloc ; `continue` ignore les événements hors authentification.
- Un succès (`channel == "auth"`) remet la série de cette IP à zéro. Filtrer seulement sur `auth_failure` ne le permettrait pas, c'est pourquoi les deux canaux sont écoutés.
- La clause `else` du `for` ne s'exécute que si le bloc a été traité **sans `break`** : elle compte alors un cycle de détection complet. Un `break` (option `max_alerts`) marque l'analyse comme interrompue (`completed = False`).

In [5]:
for alert in report.alerts:
    print(alert)
print("analyse complète :", report.completed, "| cycles :", report.cycles_completed)

BruteForceAlert(ip='10.0.0.5', first_seen=datetime.datetime(2026, 11, 3, 8, 1, 10), last_seen=datetime.datetime(2026, 11, 3, 8, 1, 18), attempts=5, users={'root'})
analyse complète : True | cycles : 1


In [6]:
# Arrêt anticipé : le flux n'est pas consommé jusqu'au bout
early = analyze(transform(parse_logs(read_multiple_logs(paths))), chunk_size=1, max_alerts=1)
print("completed =", early.completed, "| entrées lues =", early.entries_scanned)

completed = False | entrées lues = 7


## 4. Gestion mémoire

L'entrée est n'importe quel itérable (générateur compris). Seuls les agrégats sont conservés : leur taille dépend du nombre d'IP, d'utilisateurs et de canaux distincts, pas du nombre de lignes. Le test `test_early_stop_is_lazy` vérifie que seules 5 entrées sur 1000 sont consommées lors d'un arrêt anticipé.

## 5. Tests

59 tests `unittest` passent (`python -m unittest -v`), dont 26 propres à ce module.